In [0]:
dbutils.widgets.text("catalog_name", "workspace")
dbutils.widgets.text("schema_name", "bronze")
dbutils.widgets.text("base_path", "/Volumes/workspace/raw_zone/chinook")
dbutils.widgets.text("table_name", "all")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name  = dbutils.widgets.get("schema_name")
base_path    = dbutils.widgets.get("base_path")
table_name   = dbutils.widgets.get("table_name")

print("=" * 50)
print("PIPELINE PARAMETERS")
print("=" * 50)
print(f"Catalog   : {catalog_name}")
print(f"Schema    : {schema_name}")
print(f"Base Path : {base_path}")
print(f"Table     : {table_name}")
print("=" * 50)

Active tables from metadata

In [0]:
active_tables = spark.sql(f"""
    SELECT table_name
    FROM {catalog_name}.{schema_name}.pipeline_control
    WHERE active_flag = 'Y'
    ORDER BY table_name
""")

print("=" * 50)
print("TABLES TO LOAD INTO BRONZE")
print("=" * 50)
active_tables.show()
print(f"Total: {active_tables.count()} tables")

read from raw write to bronze

In [0]:
success_tables = []
failed_tables  = []

print("=" * 50)
print("STARTING RAW TO BRONZE LOAD")
print("=" * 50)

for row in active_tables.collect():
    tbl = row["table_name"]  # PascalCase e.g. Album, InvoiceLine

    try:
        print(f"\n📥 Processing: {tbl}")

        # Step 1 — Get the latest successful file path from execution_log
        latest = spark.sql(f"""
            SELECT file_location
            FROM {catalog_name}.{schema_name}.execution_log
            WHERE table_name = '{tbl}'
              AND status = 'SUCCESS'
            ORDER BY execution_time DESC
            LIMIT 1
        """).collect()

        if not latest:
            raise Exception(f"No successful Raw file found in execution_log for {tbl}")

        file_path = latest[0]["file_location"]
        print(f"   Reading from : {file_path}")

        # Step 2 — Read the Raw Parquet file
        df = spark.read.parquet(file_path)
        raw_count = df.count()
        print(f"   Raw rows     : {raw_count}")

        # Step 3 — Write to Bronze as Delta — overwrite every run
        # Bronze always holds today's snapshot — no transformations
        bronze_table = f"{catalog_name}.{schema_name}.{tbl}"
        df.write.format("delta").mode("overwrite") \
          .saveAsTable(bronze_table)

        # Step 4 — Verify
        bronze_count = spark.table(bronze_table).count()
        status = "SUCCESS" if raw_count == bronze_count else "FAILED"
        print(f"   Bronze rows  : {bronze_count}")
        print(f"   Status       : {status}")

        success_tables.append(tbl)
        print(f"   ✅ Bronze table ready: {bronze_table}")

    except Exception as e:
        failed_tables.append(tbl)
        print(f"   ❌ FAILED | {str(e)}")

validating tables in bronze

In [0]:
print("Validating Bronze tables in Unity Catalog...")
spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}").show(truncate=False)

Validate: spot check Album Bronze table

In [0]:
print("Spot check — Bronze Album table:")
spark.sql(f"""
    SELECT * FROM {catalog_name}.{schema_name}.Album LIMIT 5
""").show()

bronze_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {catalog_name}.{schema_name}.Album").collect()[0]["cnt"]
print(f"Bronze Album row count: {bronze_count}")
print("✅ Should match Raw count of 347")